# 0. Start here: the toric code in three moves

This notebook is for readers who know what a qubit and the Pauli matrices $X, Y, Z$ are, and have heard words like
*topological order*, *anyons* and *braiding* without them clicking. The fastest way in is to play:
open [`app/toric_code_playground.html`](../app/toric_code_playground.html) in a browser, click an edge, and watch.
Everything the app shows is computed by the same rules as the small Python package used here, `toric`.

**The model.** One qubit sits on every edge of an $L \times L$ square grid whose opposite sides are glued: a torus.

$$H = -\sum_v A_v - \sum_p B_p, \qquad A_v = \prod_{e \ni v} X_e, \qquad B_p = \prod_{e \in p} Z_e .$$

$A_v$ (a *star*) acts on the 4 edges at a vertex, $B_p$ (a *plaquette*) on the 4 edges around a square. They all
commute and each is $\pm 1$, so a ground state has every one of them equal to $+1$.

In [1]:
import matplotlib.pyplot as plt
import numpy as np

import toric

BLUE, ORANGE = "#2a78d6", "#eb6834"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.figsize": (5.5, 3.4)})

## Move 1: one gate makes two particles

Apply a single $Z$ to one edge of a ground state. It anticommutes with the two stars at the ends of that edge, so
both flip to $-1$. Each flipped star costs energy $+2$ and behaves like a particle, called an **e anyon**.

In [2]:
s = toric.new_state(6)            # 6 x 6 torus, 72 qubits, in a ground state
t = s.torus
print("ground state:  E =", s.energy())

s.apply("Z", t.h(2, 2))           # Z on the horizontal edge leaving vertex (2, 2)
e, m = s.anyons()
print("after one Z:   E =", s.energy(), " e anyons at", e)

ground state:  E = -72
after one Z:   E = -68  e anyons at [(2, 2), (3, 2)]


Anyons always come in pairs out of nothing. Now push one of them: a $Z$ on the next edge flips that star back and
flips the next one. The anyon moves, and **the energy does not change**: only the two ends of the string of $Z$'s
cost energy, never the string itself.

In [3]:
for step in range(3):
    s.apply("Z", t.h(3 + step, 2))
    print(f"string of {step + 2} Z's:  E = {s.energy()},  e anyons at {s.anyons()[0]}")

string of 2 Z's:  E = -68,  e anyons at [(2, 2), (4, 2)]
string of 3 Z's:  E = -68,  e anyons at [(2, 2), (5, 2)]
string of 4 Z's:  E = -68,  e anyons at [(0, 2), (2, 2)]


## Move 2: drag a particle around the torus and store a bit

Close a string into a loop that wraps all the way around the torus. No anyons are left and the energy is back to
the ground-state value, yet the state is **different**: the torus has 4 ground states, and loops around it are what
tell them apart. Those 4 states are two qubits, $|q_1 q_2\rangle$, stored in loops that no single edge can see.

In [4]:
s = toric.new_state(6)
s.apply_string("X", t.x_col(2))   # X across a whole column: an m anyon dragged around the torus
print("anyons:", s.anyons(), " E =", s.energy())
print("logical state, starting from |00>:", np.round(s.logical([1, 0, 0, 0]), 3), "  (that is |10>)")

anyons: ([], [])  E = -72
logical state, starting from |00>: [0.+0.j 0.+0.j 1.+0.j 0.+0.j]   (that is |10>)


## Move 3 (the failure case): heat scrambles the stored qubits

At temperature $T$, random errors create anyon pairs at a rate $\propto e^{-4/T}$. The pairs wander for free and
eventually scramble the two qubits. *Decoding* (joining the anyons back up in pairs) helps only if it is done often.
Below: the number of sweeps (one attempted error per edge each) until the stored state is lost.

In [5]:
rng = np.random.default_rng(0)
for every in (None, 1):            # never correct, or decode after every sweep
    times = [toric.memory_lifetime(6, 0.6, every, 2000, rng) for _ in range(10)]
    print(f"{'correct every sweep' if every else 'no correction      '}: lost after ~{np.mean(times):.0f} sweeps")

no correction      : lost after ~14 sweeps
correct every sweep: lost after ~215 sweeps


Without correction the memory is lost quickly, and a bigger torus does **not** help. With correction it survives
much longer, and longer still for bigger tori. Notebook 3 shows why.

## Where to go next

| Step | What you learn | Notebook |
|---|---|---|
| 1 | Anyons in the wavefunction: the ground state as a loop soup, e as a sign twist, m as a loop end, braiding = $-1$ | [`01_anyons`](01_anyons.ipynb) |
| 2 | Two logical qubits: reading them with loops, gates as anyons going around the torus, what braiding cannot do | [`02_logical_qubits`](02_logical_qubits.ipynb) |
| 3 | Heat: thermal anyons, why a lone pair does not simply annihilate, and why protection needs error correction | [`03_thermal_memory`](03_thermal_memory.ipynb) |

## Exercises

1. Make an e pair, then move one e all the way around the torus back to its partner. Is `s.logical([1, 0, 0, 0])`
   changed? Try `s.logical([1, 0, 1, 0] / np.sqrt(2))` instead. (Notebook 2 explains the difference.)
2. Apply $X$ on one edge. Where are the anyons now, and what kind are they (`s.anyons()` returns `(e, m)`)?
3. Rerun the last cell with `L = 4` and `L = 10`. Which column changes, and which does not?